<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>

# Physical Duckiedrone Access

Having the ability to access remotely a device, like a Duckietown robot, is extremely useful, for example to inspect files or running programs on it. The standard way to do so is by using Secure Shell (SSH), with the `ssh` command.

SSH opens an encrypted remote shell from your base station. You can then use the Linux commands you have already learned on the Duckiedrone, without attaching a keyboard or screen.

In this notebook, you will connect to your robot, verify where commands run, and return to the base station. An optional final step configures a personal key for future logins.

SSH gives you the permissions of the account you log into—not automatically unrestricted access. Depending on the robot’s configuration, you can authenticate with a password or a personal key; creating a key pair is not always required before the first connection.

## Verify the intended Duckiedrone first

Ssh'ing is a delicate operation, because it gives complete access to a device from an external source. When setting up `ssh` connections we therefore want to put particular attention to security. From a base-station terminal connected to a network you own, or are explicitly authorized to administer, verify only a Duckiedrone you own or are explicitly authorized to access. Trying to use third-party networks to `ssh` into devices that do not belong to us (or are not authorized to `ssh` into) can be interpreted as a hostile cyber action. 

Before trying to `ssh` into a device (from now on we will assume it is a Duckiedrone, but this works with virtually any computer), we need to make sure that there is an established connection between our base-station (the device we're ssh-ing from) to the Duckiedrone (the device we're ssh-ing into). 

First, make sure your base station and Duckiedrone are on the same network. Refer to the Duckietown Manual for [how to configure the network on a Duckiedrone](https://docs.duckietown.com/ente/opmanual-dd24/30-duckiedrone-setup-and-config/20-first-connection.html). 

You should know the Duckiedrone's hostname/robotname, let's call it `DUCKIEDRONE_NAME` already. If not, ask your instructor, or refer to the [Duckietown operation manual](https://docs.duckietown.com/ente/opmanual-dd24/30-duckiedrone-setup-and-config/10-first-boot.html).

To verify that the base station and robot can reach each other on the network:

```bash
ping -c 3 DUCKIEDRONE_NAME.local
```

A successful ping shows that the Duckiedrone responded to ICMP echo, which is a great starting point. The next step is to verify that an SSH service is available.

## Open an authorized remote session

From the same base-station terminal, run:

```shell
ssh duckie@DUCKIEDRONE_NAME.local
```

At the first connection, SSH displays a host-key fingerprint, a short identifier for the Duckiedrone's cryptographic identity. Something like this:

```shell
The authenticity of host 'DUCKIEDRONE_NAME.local (...)' can't be established.
ED25519 key fingerprint is SHA256:...
Are you sure you want to continue connecting (yes/no/[fingerprint])?
```

The address and fingerprint are abbreviated here. Compare the complete fingerprint shown in your terminal with the trusted setup record. Enter yes only when they match. If you have no trusted fingerprint, obtain it from the device owner or instructor. 

SSH normally saves the accepted host key in your base station’s `~/.ssh/known_hosts` file. This lets it recognize the server on later connections. An unexpected change may follow reinstallation, but needs verification before you accept it. See [OpenSSH’s host-key checking documentation](https://man.openbsd.org/ssh_config.5#StrictHostKeyChecking) for additional detail.

You will then be prompted for the user's (`duckie`) password on the remote device (`DUCKIEDRONE_NAME` = `amelia` in the example below):

```shell
duckie@amelia.local's password:
```

Enter the password configured for that robot account. Note that **no characters or asterisks appear while you type**, that is normal. Press `Enter` when finished. If an authorized personal key is already configured, you may not receive a robot-password prompt (it is possible to setup `ssh` so that the password step is bypassed).

After the session opens, you know if it succeeded because your prompt will look something like this: 

```shell
duckie@amelia:~ $
```

Congratulations, you are actually on the remote machine! To get your bearings, run:

```shell
hostname
whoami
pwd
```

You should get:

```shell
amelia
duckie
/home/duckie
```

These identify the computer, account, and current directory. From this point, commands entered in this terminal operate on. For example, `ls` lists files on the robot, and `localhost` refers to the robot’s network context.

```text
Base-station shell → ssh → Duckiedrone shell
                   ← exit ←
```

To close the remote terminal and return to base station, type `exit`. Run `hostname` again to confirm you are back on your base station. Note that leaving the remote session does not shut down the robot. [OpenSSH’s ssh manual](https://man.openbsd.org/ssh.1) describes remote sessions and authentication.


#### If login fails

There are several reasons for which `ssh` could not work, for example:

| Message | What to investigate |
| --- | --- |
| `Could not resolve hostname` | The hostname and name-resolution setup |
| `Connection refused` or a timeout | The network path, SSH service, and access rules |
| `Permission denied` | The username and accepted password or key |
| Host-key change warning | Whether the device’s identity changed as expected |

For example, `Permission denied` after password entry means you reached an SSH server but authentication failed. Repeating network discovery will not correct the account credentials.

## Install one personal public key on the robot for password-less access

For repeated connections, a personal key can replace entering the robot’s account password. This is separate from the host key checked above:

| Key | Purpose |
| --- | --- |
| Robot’s host key | Identifies the SSH server |
| Your personal key pair | Authenticates you to the robot account |

Your **private key** stays on the base station. Its matching **public key** can be installed on the robot.

This step is optional and changes the account’s future access configuration. Use it when your device setup permits personal keys and a password login has already worked.

Back in the **base-station terminal**, check for your existing public key:

```bash
ls ~/.ssh/id_ed25519.pub
```

If it is missing, follow your course’s key-creation instructions before continuing; skip this optional step if none are provided. If your personal key uses another filename, use that path instead.

Once created, install the public key on the robot:

```bash
ssh-copy-id -i ~/.ssh/id_ed25519.pub "duckie@DUCKIEDRONE_NAME.local"
```

This command authenticates to the robot and adds the selected public key to the account’s authorized keys. It normally requests the robot password if the key is not already accepted. See the [`ssh-copy-id` manual](https://man7.org/linux/man-pages/man1/ssh-copy-id.1.html).

You can then confirm success by reconnecting. From your base station:

```bash
ssh "duckie@DUCKIEDRONE_NAME.local"
```

A key may be protected by a **passphrase**, which unlocks the private key locally. This password, if it exists, has been defined during the ssh keypair generation process and has nothing to do with the robot/user password. Never copy the private key to the robot or include it in shared output, or anyone will be able to authenticate as you. 

Run `hostname` to confirm the destination, then `exit` when finished.

SSH is a powerful tool that we will leverage extensively goin forward. 


## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
